RetailPulse 360

Goal: Extract transferable retail store behavior from Rossmann without transferring its absolute sales scale. These behavioral traits will later be used to seed realistic demand patterns for simulated Stylo stores.

Output: store_personalities.csv

In [1]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# 2. LOAD DATA
# ============================================================

TRAIN_PATH = "/kaggle/input/datasets/hamaz911/rossman/train.csv"
STORE_PATH = "/kaggle/input/datasets/hamaz911/rossman/store.csv"

train = pd.read_csv(
    TRAIN_PATH,
    low_memory=False,
    parse_dates=["Date"]
)

store = pd.read_csv(STORE_PATH)

print("Data loaded successfully.")

Data loaded successfully.


In [3]:
# 3. INITIAL DATA INSPECTION
# ============================================================

print("TRAIN DATA")
print("Shape:", train.shape)
print("Date range:", train["Date"].min(), "to", train["Date"].max())
print("Unique stores:", train["Store"].nunique())

print("\nSTORE DATA")
print("Shape:", store.shape)
print("Unique stores:", store["Store"].nunique())

print("\nFirst 5 training rows:")
display(train.head())

print("\nFirst 5 store rows:")
display(store.head())

TRAIN DATA
Shape: (1017209, 9)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
Unique stores: 1115

STORE DATA
Shape: (1115, 10)
Unique stores: 1115

First 5 training rows:


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1



First 5 store rows:


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [4]:
# 4. DATA QUALITY — DUPLICATE STORE-DATE RECORDS
# ============================================================

dupes = train.duplicated(
    subset=["Store", "Date"]).sum()

print("Duplicate Store-Date rows:", dupes)

assert dupes == 0, (
    "Found duplicate Store-Date rows — "
    "must resolve before proceeding."
)

print("No duplicate Store-Date records found.")

Duplicate Store-Date rows: 0
No duplicate Store-Date records found.


In [5]:
# 5. DATA QUALITY — MISSING VALUES
# ============================================================

print("TRAIN.CSV MISSING VALUES")
print("-" * 50)

train_missing = train.isna().sum()
train_missing = train_missing[train_missing > 0]

print(train_missing if len(train_missing) > 0 else "None")

print("\nSTORE.CSV MISSING VALUES")
print("-" * 50)

store_missing = store.isna().sum()
store_missing = store_missing[store_missing > 0]

print(store_missing if len(store_missing) > 0 else "None")

TRAIN.CSV MISSING VALUES
--------------------------------------------------
None

STORE.CSV MISSING VALUES
--------------------------------------------------
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [6]:
# 6. DATA QUALITY — CALENDAR CONSISTENCY
# ============================================================

computed_dow = train["Date"].dt.dayofweek + 1

mismatch = (
    computed_dow != train["DayOfWeek"]
).sum()

print(
    "Rows where DayOfWeek disagrees with the actual "
    f"calendar weekday: {mismatch}"
)

assert mismatch == 0, (
    "DayOfWeek labels do not match actual dates — "
    "must resolve before proceeding."
)

print("✓ Calendar weekday labels are consistent.")

Rows where DayOfWeek disagrees with the actual calendar weekday: 0
✓ Calendar weekday labels are consistent.


In [7]:
# 7. FILTER OPEN STORE DAYS
# ============================================================

open_days = train[
    train["Open"] == 1
].copy()

print("Total training rows:", len(train))
print("Open-day rows:", len(open_days))
print(
    "Open-day percentage:",
    round(len(open_days) / len(train) * 100, 2),
    "%"
)

Total training rows: 1017209
Open-day rows: 844392
Open-day percentage: 83.01 %


In [8]:
# 8. DATA QUALITY — OPEN-DAY SALES ANOMALIES
# ============================================================

q1, q3 = open_days["Sales"].quantile([0.25, 0.75])
iqr = q3 - q1

upper_bound = q3 + 3 * iqr

extreme = (
    open_days["Sales"] > upper_bound
).sum()

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Extreme-outlier threshold: {upper_bound:.0f}")

print(
    f"Rows above threshold: {extreme} "
    f"({extreme / len(open_days) * 100:.3f}%)"
)

Q1: 4859.00
Q3: 8360.00
IQR: 3501.00
Extreme-outlier threshold: 18863
Rows above threshold: 5998 (0.710%)


In [9]:
# 9. DATA QUALITY — OPEN-DAY ZERO SALES
# ============================================================

anomaly = open_days[
    open_days["Sales"] == 0
]

print("Open-day rows with zero sales:", len(anomaly))
print(
    "Unique stores affected:",
    anomaly["Store"].nunique()
)

print("\nMost common dates:")
display(
    anomaly["Date"]
    .value_counts()
    .head(5)
)

print("\nCustomer statistics:")
display(
    anomaly["Customers"].describe()
)

Open-day rows with zero sales: 54
Unique stores affected: 41

Most common dates:


Date
2014-07-24    4
2014-07-23    2
2014-09-11    2
2015-05-15    1
2015-03-26    1
Name: count, dtype: int64


Customer statistics:


count    54.000000
mean      0.148148
std       0.786859
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       5.000000
Name: Customers, dtype: float64

In [10]:
# 10. INITIALIZE STORE PERSONALITY EXTRACTION
# ============================================================

profiles = []

MIN_OPEN_DAYS = 100

print(
    f"Minimum required open days per store: {MIN_OPEN_DAYS}"
)

Minimum required open days per store: 100


In [11]:
# 11. EXTRACT STORE PERSONALITY FEATURES
# ============================================================

for store_id, g in open_days.groupby("Store"):

    # Sort chronologically for time-based calculations
    g = g.sort_values("Date").copy()

    # Store-specific baseline
    mean_sales = g["Sales"].mean()

    # Skip stores with insufficient history
    if mean_sales == 0 or len(g) < MIN_OPEN_DAYS:
        continue

    # --------------------------------------------------------
    # 1. Day-of-week rhythm
    # --------------------------------------------------------

    dow = (
        g.groupby("DayOfWeek")["Sales"].mean()
        / mean_sales
    )

    # --------------------------------------------------------
    # 2. Promotion response
    # --------------------------------------------------------

    promo_mean = g.loc[
        g["Promo"] == 1,
        "Sales"
    ].mean()

    nonpromo_mean = g.loc[
        g["Promo"] == 0,
        "Sales"
    ].mean()

    promo_lift = (
        promo_mean / nonpromo_mean - 1
        if nonpromo_mean > 0 and not np.isnan(nonpromo_mean)
        else 0
    )

    # --------------------------------------------------------
    # 3. Long-term trend
    # --------------------------------------------------------

    t = (
        g["Date"] - g["Date"].min()
    ).dt.days.values

    slope = (
        np.polyfit(
            t,
            g["Sales"].values,
            1
        )[0]
        if len(g) > 1
        else 0
    )

    trend_pct_per_year = (
        slope * 365
    ) / mean_sales

    # --------------------------------------------------------
    # 4. Demand volatility
    # --------------------------------------------------------

    volatility_cv = (
        g["Sales"].std() / mean_sales
    )

    # --------------------------------------------------------
    # 5. Holiday sensitivity
    # --------------------------------------------------------

    holiday_mean = g.loc[
        g["SchoolHoliday"] == 1,
        "Sales"
    ].mean()

    normal_mean = g.loc[
        g["SchoolHoliday"] == 0,
        "Sales"
    ].mean()

    holiday_lift = (
        holiday_mean / normal_mean - 1
        if normal_mean > 0 and not np.isnan(normal_mean)
        else 0
    )

    # --------------------------------------------------------
    # Store personality record
    # --------------------------------------------------------

    profiles.append({
        "rossmann_store_id": store_id,

        "dow_mon": dow.get(1, 1.0),
        "dow_tue": dow.get(2, 1.0),
        "dow_wed": dow.get(3, 1.0),
        "dow_thu": dow.get(4, 1.0),
        "dow_fri": dow.get(5, 1.0),
        "dow_sat": dow.get(6, 1.0),
        "dow_sun": dow.get(7, 1.0),

        "promo_lift": round(promo_lift, 4),
        "trend_pct_per_year": round(
            trend_pct_per_year,
            4
        ),
        "volatility_cv": round(
            volatility_cv,
            4
        ),
        "holiday_lift": round(
            holiday_lift,
            4
        )
    })

print(
    "Store personality extraction completed."
)

Store personality extraction completed.


In [12]:
# 12. CREATE STORE PERSONALITY DATAFRAME
# ============================================================

profiles_df = pd.DataFrame(profiles)

print(
    "Personality profiles extracted:",
    len(profiles_df)
)

print(
    "Number of personality features:",
    profiles_df.shape[1]
)

display(
    profiles_df.head()
)

Personality profiles extracted: 1115
Number of personality features: 12


,rossmann_store_id,dow_mon,dow_tue,dow_wed,dow_thu,dow_fri,dow_sat,dow_sun,promo_lift,trend_pct_per_year,volatility_cv,holiday_lift
0,1,1.088015,0.984562,0.957264,0.936699,0.993147,1.038636,1.0,0.2269,-0.0330,0.2127,0.0099
1,2,1.223673,1.083833,1.177849,1.002740,0.942609,0.579968,1.0,0.6245,0.0288,0.3250,0.0948
2,3,1.201053,1.101155,1.027312,0.999117,1.036101,0.643564,1.0,0.6450,0.0018,0.3159,0.0813
3,4,1.125869,0.976293,0.925463,0.941326,0.982571,1.049287,1.0,0.1923,0.0390,0.2009,0.0641
4,5,1.316537,1.092685,1.114581,1.015330,1.038873,0.444607,1.0,0.7363,0.0172,0.3776,0.0766


In [13]:
# 13. PERSONALITY PROFILE SUMMARY
# ============================================================

display(
    profiles_df
    .describe()
    .round(3)
)

,rossmann_store_id,dow_mon,dow_tue,dow_wed,dow_thu,dow_fri,dow_sat,dow_sun,promo_lift,trend_pct_per_year,volatility_cv,holiday_lift
count,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000,1115.000
mean,558.000,1.186,1.022,0.970,0.974,1.021,0.837,0.995,0.414,0.039,0.272,0.045
std,322.017,0.078,0.067,0.045,0.050,0.044,0.193,0.076,0.181,0.059,0.058,0.059
min,1.000,0.958,0.845,0.835,0.854,0.874,0.323,0.262,-0.070,-0.242,0.123,-0.093
25%,279.500,1.131,0.971,0.939,0.941,0.990,0.703,1.000,0.294,0.011,0.233,0.010
50%,558.000,1.183,1.021,0.965,0.971,1.020,0.844,1.000,0.406,0.035,0.264,0.037
75%,836.500,1.234,1.067,0.996,1.001,1.049,0.982,1.000,0.518,0.064,0.301,0.069
max,1115.000,1.442,1.298,1.244,1.316,1.234,1.392,1.577,1.449,0.365,0.551,0.498


In [14]:
# 14. VALIDATE BEHAVIORAL FEATURES
# ============================================================

ratio_columns = [
    "dow_mon",
    "dow_tue",
    "dow_wed",
    "dow_thu",
    "dow_fri",
    "dow_sat",
    "dow_sun",
    "promo_lift",
    "holiday_lift"
]

print("Behavioral feature ranges:")
display(
    profiles_df[ratio_columns]
    .agg(["min", "mean", "max"])
    .round(3)
)

Behavioral feature ranges:


,dow_mon,dow_tue,dow_wed,dow_thu,dow_fri,dow_sat,dow_sun,promo_lift,holiday_lift
min,0.958,0.845,0.835,0.854,0.874,0.323,0.262,-0.070,-0.093
mean,1.186,1.022,0.970,0.974,1.021,0.837,0.995,0.414,0.045
max,1.442,1.298,1.244,1.316,1.234,1.392,1.577,1.449,0.498


In [15]:
# 15. SAVE STORE PERSONALITIES
# ============================================================

OUTPUT_PATH = "store_personalities.csv"

profiles_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Saved: {OUTPUT_PATH}"
)

print(
    f"Shape: {profiles_df.shape[0]} stores × "
    f"{profiles_df.shape[1]} features"
)

Saved: store_personalities.csv
Shape: 1115 stores × 12 features


In [16]:
# 16. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 01 — STORE PERSONALITY EXTRACTION")

print(f"Rossmann stores analyzed: {train['Store'].nunique():,}")
print(f"Stores with valid personalities: {len(profiles_df):,}")
print(f"Open-day observations used: {len(open_days):,}")
print(f"Output file: {OUTPUT_PATH}")

print("\nBehavioral traits extracted:")
print("• Day-of-week rhythm")
print("• Promotion response")
print("• Long-term trend")
print("• Demand volatility")
print("• Holiday sensitivity")

print("\n✓ Notebook 01 completed successfully.")

NOTEBOOK 01 — STORE PERSONALITY EXTRACTION
Rossmann stores analyzed: 1,115
Stores with valid personalities: 1,115
Open-day observations used: 844,392
Output file: store_personalities.csv

Behavioral traits extracted:
• Day-of-week rhythm
• Promotion response
• Long-term trend
• Demand volatility
• Holiday sensitivity

✓ Notebook 01 completed successfully.


In [17]:
# 17. ZIP ARTIFACT FOR LOCAL DOWNLOAD
# ============================================================

import shutil
from pathlib import Path

OUTPUT_FILE = Path("store_personalities.csv")
ZIP_NAME = "notebook_01_store_personality_artifact"

if OUTPUT_FILE.exists():
    zip_path = shutil.make_archive(
        ZIP_NAME,
        "zip",
        root_dir=".",
        base_dir=OUTPUT_FILE.name
    )

    print("ZIP created successfully:")
    print(zip_path)
    print(f"ZIP size: {Path(zip_path).stat().st_size / 1024:.2f} KB")
else:
    raise FileNotFoundError(
        f"{OUTPUT_FILE} was not found. "
        "Run the previous save cell first."
    )

ZIP created successfully:
/kaggle/working/notebook_01_store_personality_artifact.zip
ZIP size: 73.58 KB
